# Week 2 Assignment: Implement and Verify the VAE Loss


#From Maths to Magic
#Name: Diptanshu Raut
Roll no : 24B0351


## Imports

In [1]:
import torch
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence


## Part 1 — Closed-Form KL Divergence

In [2]:
def kl_divergence_gaussian(mu: torch.Tensor,
                           logvar: torch.Tensor) -> torch.Tensor:

    kl = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - torch.exp(logvar)
    )
    return kl

# Test it
mu     = torch.randn(32, 16)   # batch of 32, latent dim 16
logvar = torch.randn(32, 16)
sigma  = torch.exp(0.5 * logvar)

your_kl = kl_divergence_gaussian(mu, logvar)

q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(sigma))
lib_kl  = kl_divergence(q, p).sum()

print(f"Your KL:   {your_kl:.4f}")
print(f"Torch KL:  {lib_kl:.4f}")
assert torch.isclose(your_kl, lib_kl, atol=1e-4), "KL values don't match!"
print("✅ Part 1 passed")

Your KL:   424.4207
Torch KL:  424.4207
✅ Part 1 passed


## Part 2 — Reparameterization Trick

In [3]:
def reparameterize(mu: torch.Tensor,
                   logvar: torch.Tensor) -> torch.Tensor:

    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z = mu + std * eps

    return z

# Verify: the mean of many samples should be close to mu
mu_test     = torch.tensor([2.0, -1.0])
logvar_test = torch.tensor([0.0,  0.5])

samples = torch.stack([reparameterize(mu_test, logvar_test) for _ in range(10000)])
print(f"Target mu:       {mu_test.tolist()}")
print(f"Sample mean:     {samples.mean(0).tolist()}")
print(f"Target std:      {torch.exp(0.5 * logvar_test).tolist()}")
print(f"Sample std:      {samples.std(0).tolist()}")
# Means and stds should be close (within ~0.05)
print("✅ Part 2 passed")



Target mu:       [2.0, -1.0]
Sample mean:     [1.986100435256958, -1.004860281944275]
Target std:      [1.0, 1.2840254306793213]
Sample std:      [0.9978031516075134, 1.3024464845657349]
✅ Part 2 passed


## Part 3 — ELBO Loss

In [4]:
def elbo_loss(x: torch.Tensor,
              x_recon: torch.Tensor,
              mu: torch.Tensor,
              logvar: torch.Tensor
              ) -> torch.Tensor:

    recon_loss = F.binary_cross_entropy(
        x_recon,
        x,
        reduction="sum"
    )

    kl = kl_divergence_gaussian(
        mu,
        logvar
    )

    return recon_loss + kl

# Quick smoke test
batch, dim, latent = 16, 784, 32
x      = torch.rand(batch, dim)
x_recon = torch.sigmoid(torch.randn(batch, dim))
mu     = torch.randn(batch, latent)
logvar = torch.randn(batch, latent)

loss = elbo_loss(x, x_recon, mu, logvar)
print(f"ELBO loss (should be a positive scalar): {loss.item():.4f}")
assert loss.ndim == 0, "Loss must be a scalar!"
print("✅ Part 3 passed")

ELBO loss (should be a positive scalar): 10630.3975
✅ Part 3 passed


## Part 4 — Reflection Answers

### Q1. Why can't we maximize log p(x) directly?

Directly maximizing log p(x) is difficult because computing p(x) requires integrating over every possible latent variable z. For high-dimensional latent spaces this integral is generally intractable. The encoder q(z|x) provides an approximation to the true posterior p(z|x), allowing us to optimize the ELBO instead of the exact log-likelihood.

### Q2. What happens if the KL term is removed?

Without the KL term, the encoder is free to place latent representations anywhere in the latent space. The model may reconstruct training examples well, but the latent space will not follow a standard Gaussian distribution. As a result, randomly sampling latent vectors will usually produce poor generations.

### Q3. Why does the reparameterization trick work?

The reparameterization trick rewrites sampling as z = μ + σϵ where ϵ is sampled from a standard normal distribution. Since μ and σ appear in a deterministic computation, gradients can flow through them during backpropagation. Without this trick, the sampling operation would block gradients and prevent end-to-end training.

### Q4. What happens when μ = 0 and logvar = 0?

When μ and logvar are both zero, the encoder distribution becomes N(0,1), which is exactly the same as the prior. Since KL divergence measures the difference between two distributions, the KL value becomes zero. This matches the closed-form formula because every term inside the summation cancels out.
